# pplx-embed-v2 late 0.6B — search page images without OCR

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/pplx-embed-v2-late-0.6b-page-search.ipynb)

[`perplexity-ai/pplx-embed-v2-late-0.6b`](https://huggingface.co/perplexity-ai/pplx-embed-v2-late-0.6b) is a multimodal late-interaction retriever (ColBERT-style, MIT licence). It emits one 128-dimensional vector per token and scores a query against a page with MaxSim, so a text question can hit a PDF page or a slide screenshot directly. There is no OCR step and no single pooled vector. The 0.6B and 9B checkpoints share an embedding space; this notebook uses the 0.6B, which the card lists at 340M active parameters. The Hub file is float32 and about 2.4 GB.

**Licence:** MIT.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has no native bfloat16 and no FlashAttention 2, so the notebook loads the tower in **float16** with **SDPA**. Float16 weights are about 1.2 GB; encoding a few page images peaks at roughly 3–5 GB. Text and images must be encoded in separate calls. Mixed batches are not supported, and the query/document marker belongs at the start of the text (`[Q] ` / `[D] `), which is how this Sentence Transformers export applies it. PyLate's second-position marker is the wrong format for this checkpoint.

**Links:** [model card](https://huggingface.co/perplexity-ai/pplx-embed-v2-late-0.6b) · [blog](https://www.perplexity.ai/hub/blog/multimodal-embeddings-beyond-a-single-vector)


In [ ]:
%pip install -q "sentence-transformers==6.1.0" "transformers==5.20.1" pillow pypdfium2==5.14.0

# The model card requires sentence-transformers>=6.0.0 and transformers>=5.4.0,<6.
# MultiVectorEncoder, and this checkpoint's modules.json, land in that range.
# 6.1.0 / 5.20.1 are pinned releases inside it.


> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported transformers. Skipping the restart leaves the old package in memory.


In [ ]:
import torch
import transformers
from packaging.version import Version
import sentence_transformers

assert Version(transformers.__version__) >= Version("5.4.0")
assert Version(sentence_transformers.__version__) >= Version("6.0.0")

if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, _minor = torch.cuda.get_device_capability()
DTYPE = torch.bfloat16 if major >= 8 else torch.float16
print("sentence-transformers", sentence_transformers.__version__)
print("transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(), "| dtype:", DTYPE)
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


## 1. Load the late-interaction encoder

`MultiVectorEncoder` is the class the model card uses. The saved prompts are `query: "[Q] "` and `document: "[D] "`, and the similarity function is MaxSim. `model_kwargs` is forwarded to the Hugging Face tower so the T4 load is float16 + SDPA rather than the checkpoint's float32 default.


In [ ]:
from sentence_transformers import MultiVectorEncoder

MODEL_ID = "perplexity-ai/pplx-embed-v2-late-0.6b"


def load_encoder(attn):
    return MultiVectorEncoder(
        MODEL_ID,
        device="cuda",
        model_kwargs={"dtype": DTYPE, "attn_implementation": attn},
    )


try:
    model = load_encoder("sdpa")
    print("attention: sdpa")
except Exception as exc:
    print("sdpa load failed, retrying eager:", exc)
    model = load_encoder("eager")
    print("attention: eager")

print("prompts:", getattr(model, "prompts", None))
print("similarity:", getattr(model, "similarity_fn_name", None))
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


## 2. Index three paper pages

Each document is the first page of a public arXiv PDF, rendered and capped at 1280 px on the longest edge. Pages are encoded one at a time, image-only, which is the card's rule (no text mixed into an image batch).


In [ ]:
import io

import requests
from PIL import Image

PAPERS = [
    ("Attention Is All You Need", "https://arxiv.org/pdf/1706.03762"),
    ("BERT", "https://arxiv.org/pdf/1810.04805"),
    ("Language Models are Few-Shot Learners (GPT-3)", "https://arxiv.org/pdf/2005.14165"),
]
UA = {"User-Agent": "donvito-notebooks/1.0 (educational Colab demo)"}


def first_page(url, longest=1280):
    import pypdfium2 as pdfium

    response = requests.get(url, headers=UA, timeout=120)
    response.raise_for_status()
    pdf = pdfium.PdfDocument(response.content)
    image = pdf[0].render(scale=150 / 72).to_pil().convert("RGB")
    image.thumbnail((longest, longest))
    return image


pages = [(title, first_page(url)) for title, url in PAPERS]
for title, image in pages:
    print(f"{title}: {image.size}")


In [ ]:
def one_image(image):
    # encode_document expects a batch. A single PIL image would be read as one example
    # and would not line up with a list of query embeddings below.
    encoded = model.encode_document([image])
    return encoded[0] if isinstance(encoded, list) else encoded


documents = [one_image(image) for _title, image in pages]
print("first page tokens x dim:", tuple(documents[0].shape))


## 3. Ask in text, rank the pages

`encode_query` applies the `[Q] ` prefix. MaxSim sums, over each query token, its best match among the page tokens. The winning page is shown under the scores.


In [ ]:
import matplotlib.pyplot as plt

QUERIES = [
    "scaled dot product attention and multi-head attention",
    "bidirectional transformers pretrained with a masked language model",
    "few-shot learners and in-context learning",
]

queries = model.encode_query(QUERIES)
if not isinstance(queries, list):
    queries = [queries[i] for i in range(queries.shape[0])]

scores = model.similarity(queries, documents)
if not isinstance(scores, torch.Tensor):
    scores = torch.tensor(scores)
scores = scores.detach().float().cpu()
print(scores)

fig, axes = plt.subplots(len(QUERIES), len(pages), figsize=(12, 10))
for row, query in enumerate(QUERIES):
    winner = int(scores[row].argmax())
    print(f"\nQ: {query}")
    order = torch.argsort(scores[row], descending=True).tolist()
    for rank, col in enumerate(order, start=1):
        print(f"  {rank}. {pages[col][0]}  MaxSim={scores[row, col].item():.3f}")
    for col, (title, image) in enumerate(pages):
        ax = axes[row, col]
        ax.imshow(image)
        ax.set_axis_off()
        mark = "  << best" if col == winner else ""
        ax.set_title(f"{scores[row, col].item():.2f}{mark}\n{title}", fontsize=8)
fig.suptitle("Text query → page image (MaxSim)", fontsize=14)
fig.tight_layout()
plt.show()
